# The error taxonomy

**Scenario:** an agent supervises EV charging on one distribution feeder. It reads faults and sends
curtailment commands to hold load under the licence. One retry wrapper sits around every call.

During a load event it sent the same illegal command three times, paid three times to be told the
same nothing, and hid the one failure worth waking somebody for.

Retrying a refusal is **pressing the lift button harder**. The lift already heard you.

## Mechanics

Four kinds of failure arrive through the same `except`. Each needs its own handler, and only one is
ever the model's business.

| Tier | What it looks like | Who handles it | Action | Does the model see it? |
|---|---|---|---|---|
| infrastructure | HTTP 429, HTTP 500, a timeout | the HTTP wrapper | bounded backoff with full jitter | never |
| syntactic | the reply will not parse, a field is missing | the loop around the model | send the reason back, ask again | yes, that is the point |
| business policy | the command is above the licensed limit | the caller | stop, or hand it to a person | no, and never retry |
| empty result | the feeder has no open faults | nobody | return it as the answer | it already answered |

Backoff means waiting longer after each failed retry, so you stop making things worse. Jitter adds
randomness to that wait, so callers do not all return at once.

## The picture

![One classifier, four handlers, and only one path back to the model](images/error-taxonomy.svg)

The classifier is the only place that decides. Once four failures share a handler, that handler is
wrong for at least three of them.

## The cost

```
t = random(0, min(t_max, t_base * 2 ** attempt))
```

Full jitter. `t_base` is the first wait, `t_max` is the ceiling, and `attempt` counts from zero. The
randomness is the part people drop. Without it, every caller that failed in one second comes back in
one second, and the thing they are waiting for never recovers.

## The failure

Two calls the agent makes. The grid gateway is a stub here, because a real 429 will not arrive on
cue, and its dropped link is simulated. The model call below is real.

In [1]:
import json
from vault import get_client, load_env, model_for, Usage, cost_of

load_env()
client = get_client("09-programmatic-guardrails/02-the-error-taxonomy")

LICENSED_KW = 750
GRID = {"submitted": [], "flaky": 1}
FEEDERS = {"F-19": []}          # nothing is wrong on this feeder right now
SPEND = []


class PolicyRejected(Exception):
    """The grid operator refused the command. Sending it again sends the same
    illegal command a second time."""

The gateway call. Two very different failures leave it through the same door, which is what makes one
wrapper look reasonable.

In [2]:
def submit_curtailment(feeder_id, kw):
    """Ask the grid to hold this feeder under kw. Refuses anything above the licence."""
    if GRID["flaky"] > 0:
        GRID["flaky"] -= 1
        raise TimeoutError("grid gateway did not answer")
    GRID["submitted"].append(kw)                  # the grid logs it before judging it
    if kw > LICENSED_KW:
        raise PolicyRejected(f"{kw} kW is above the {LICENSED_KW} kW licence")
    return {"accepted_kw": kw}

The other call asks the model which feeder to query, then runs the lookup. Its default is the bug most
teams ship.

In [3]:
FAULT_TOOL = {"type": "function", "function": {
    "name": "list_ev_faults", "description": "List open charger faults on one feeder.",
    "parameters": {"type": "object",
                   "properties": {"feeder_id": {"type": "string"}},
                   "required": ["feeder_id"], "additionalProperties": False}}}


def open_faults(feeder_id, empty_is_failure=True):
    """One model call, one lookup. Returns the faults on that feeder."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200, tools=[FAULT_TOOL],
        messages=[{"role": "system", "content": "You supervise EV charging on a grid."},
                  {"role": "user", "content": f"Open charger faults on feeder {feeder_id}?"}])
    SPEND.append(Usage.from_response(reply))
    args = json.loads(reply.choices[0].message.tool_calls[0].function.arguments)
    faults = FEEDERS.get(args["feeder_id"], [])
    if empty_is_failure and not faults:
        raise ValueError(f"no faults returned for {feeder_id}")
    return faults

And the wrapper. It is careful, it is short, and it has no idea what it is retrying.

In [4]:
def retry_everything(fn, attempts=3):
    """One handler for every failure. This is the thing being taught against."""
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as error:
            print(f"  attempt {attempt + 1} failed: {type(error).__name__}")
    raise RuntimeError(f"gave up after {attempts} attempts")

Run both through it. Watch the counters rather than the messages.

In [5]:
GRID["flaky"] = 0
GRID["submitted"].clear()
try:
    retry_everything(lambda: submit_curtailment("F-19", 900))
except RuntimeError:
    pass
naive_submissions = len(GRID["submitted"])

SPEND.clear()
try:
    retry_everything(lambda: open_faults("F-19"))
except RuntimeError:
    pass
naive_calls = len(SPEND)

print(f"\nillegal command reached the grid {naive_submissions} times")
print(f"empty answer cost {naive_calls} model calls, ${sum(cost_of(u) for u in SPEND):.6f}")
assert naive_submissions == 1, f"one command, {naive_submissions} submissions"

  attempt 1 failed: PolicyRejected
  attempt 2 failed: PolicyRejected
  attempt 3 failed: PolicyRejected
  attempt 1 failed: ValueError
  attempt 2 failed: ValueError
  attempt 3 failed: ValueError

illegal command reached the grid 3 times
empty answer cost 3 model calls, $0.000029


AssertionError: one command, 3 submissions

## The diagnosis

Three attempts each, and each was wrong in its own way.

**The illegal command reached the grid three times.** Policy did not change between attempts, so the
second and third could only be refused. The audit trail now shows an operator pushing an over-licence
command repeatedly, which reads far worse than one refusal.

**The empty feeder cost three model calls.** No open faults is the correct answer. The runtime paid
for it three times, then reported a failure that never happened.

**The retry that was right is invisible.** The dropped link deserved one, and got the same three
tries as the others, with no wait between them.

The wrapper catches `Exception`. After that line, nothing knows which tier arrived.

## The fix

Name the tiers first. An enum, not a string, because a typo in a string is a silent new tier.

In [6]:
from enum import Enum


class Tier(str, Enum):
    """The four kinds of failure. Each has exactly one handler."""

    INFRASTRUCTURE = "infrastructure"
    SYNTACTIC = "syntactic"
    POLICY = "policy"
    EMPTY = "empty"

Then one place that decides. An unclassified error is raised again rather than retried, because
guessing is how the illegal command went out three times.

In [7]:
from pydantic import BaseModel, Field, ValidationError


def classify(error):
    """The only thing that decides how a failure is handled."""
    if isinstance(error, (TimeoutError, ConnectionError)):
        return Tier.INFRASTRUCTURE
    if isinstance(error, (ValidationError, json.JSONDecodeError)):
        return Tier.SYNTACTIC
    if isinstance(error, PolicyRejected):
        return Tier.POLICY
    raise error

The retry settings are configuration, so they get checked like any other input. Nobody ships
`attempts=0` if the type will not hold it.

In [8]:
class RetryPolicy(BaseModel):
    """Retry settings that cannot be misconfigured, because pydantic will not build them."""

    attempts: int = Field(ge=1, le=6)
    base_seconds: float = Field(gt=0)
    max_seconds: float = Field(gt=0)

Now the wrapper that knows what it is holding. Only the infrastructure tier waits and comes back.

In [9]:
import random
import time


def call_with_backoff(fn, policy):
    """Retry infrastructure failures with full jitter. Raise everything else at once."""
    waits = []
    for attempt in range(policy.attempts):
        try:
            return fn(), waits
        except Exception as error:
            if classify(error) is not Tier.INFRASTRUCTURE:
                raise
            ceiling = min(policy.max_seconds, policy.base_seconds * 2 ** attempt)
            waits.append(round(random.uniform(0, ceiling), 4))
            time.sleep(waits[-1])
    raise TimeoutError(f"gave up after {policy.attempts} attempts")

Same three situations, same code path, three different outcomes.

In [10]:
GRID["submitted"].clear()
GRID["flaky"] = 1
policy = RetryPolicy(attempts=4, base_seconds=0.01, max_seconds=0.2)

legal, waits = call_with_backoff(lambda: submit_curtailment("F-19", 600), policy)
print(f"link dropped, then {legal} after waits {waits}")

try:
    call_with_backoff(lambda: submit_curtailment("F-19", 900), policy)
except PolicyRejected as exc:
    print(f"illegal command refused once: {exc}")

SPEND.clear()
print(f"feeder F-19 has {len(open_faults('F-19', empty_is_failure=False))} open faults")

print(f"\nillegal commands on the audit trail: {naive_submissions} before, "
      f"{len(GRID['submitted']) - 1} after")
print(f"model calls for an empty feeder: {naive_calls} before, {len(SPEND)} after")

link dropped, then {'accepted_kw': 600} after waits [0.0027]
illegal command refused once: 900 kW is above the 750 kW licence
feeder F-19 has 0 open faults

illegal commands on the audit trail: 3 before, 1 after
model calls for an empty feeder: 3 before, 1 after


## The gate

The regression that matters is a new tier quietly joining the retry path. This test needs no model.

In [11]:
def test_only_infrastructure_is_ever_retried():
    for error in (TimeoutError("gateway"), ConnectionError("reset")):
        assert classify(error) is Tier.INFRASTRUCTURE

    for error, tier in ((PolicyRejected("over licence"), Tier.POLICY),
                        (ValidationError.from_exception_data("x", []), Tier.SYNTACTIC)):
        assert classify(error) is tier, f"{type(error).__name__} was misfiled"
        assert tier is not Tier.INFRASTRUCTURE


test_only_infrastructure_is_ever_retried()
print("gate holds: policy and syntax failures can never reach the backoff path")

gate holds: policy and syntax failures can never reach the backoff path


Add `PolicyRejected` to the first `isinstance` check and this test fails on the last line.

### Enterprise exploration

- Full jitter spreads one caller's retries. What spreads a thousand during a regional outage?
- Refused commands sit on an audit trail a regulator reads. What is the compliance cost of three
  over-licence submissions rather than one?
- The syntactic tier is the only one the model sees. What stops a failure message carrying a host name
  or a spend figure into a prompt?

### Key takeaways

- `except Exception` around a retry treats four failures as one.
- Infrastructure failures are the only ones a wait can fix. Bound them and add jitter.
- A policy refusal is an answer. Retrying it repeats the offence and delays the escalation.
- An empty result is a result. Paying three times for it is a bug you can read off the bill.